# 📌 Tabular Foundation Model

![Topic](https://img.shields.io/badge/Topic-TFM-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-architecture-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-Oct%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — A tabular foundation model is a transformer pretrained once on millions of artificial tables. At use time you hand it your labeled rows plus the rows you want predicted, and it answers immediately with no training on your data. In published benchmarks these models match or beat tuned gradient-boosted trees on small and medium datasets, but they have size limits, higher prediction cost, and still degrade when data shifts.</span>

---
## 1. Overview

<!-- What is this concept? 2–4 sentences that a newcomer could understand.
     Include: what problem it solves, why it exists, where it fits in the AI landscape. -->

A **tabular foundation model (TFM)** is a model that has already learned from a huge number of example tables, so you can use it on your own table without training anything.

**How you usually work with tables today**

Pick an algorithm such as XGBoost, LightGBM or CatBoost.
Train it on your data.
Adjust its settings (this is called tuning) and test it several times (cross-validation) to check it works well.

You repeat all of this for every new dataset.

**How a TFM works instead**

You give the model your table, with the answers you already know and the rows you want predicted. It reads everything once and returns its predictions. There is no training step and no tuning.

An analogy: a classic model is like hiring a new employee and training them from scratch for each project. A TFM is like hiring someone who has already worked on thousands of similar projects and only needs to look at your files.

**Three terms, in plain words**

* **Prior**: what the model “expects” before it sees your data, meaning which kinds of tables and patterns seem normal to it. It gets this expectation from the millions of artificial tables it practiced on.

* **In-context learning**: the model learns from your examples by reading them, like a student reading solved exercises right before an exam. Its internal settings do not change.

* **Posterior predictive distribution**: instead of giving one answer, the model gives a spread of likely answers. For example, “70% yes, 30% no”. This tells you how sure it is.

Models that work this way are called prior-data fitted networks (PFNs). TabPFN is a well-known example. It was trained once on artificial tables generated by a set of rules that mimic cause and effect between variables (Hollmann et al., 2023).

**Are TFMs better than XGBoost and similar models?**

Not always. A benchmark called TabArena found that tree-based models are still very strong overall, while TFMs do especially well on small datasets (Erickson et al., 2025).

---
## 2. How It Works

<!-- Break the concept into numbered steps or subsections.
     Use diagrams (images from assets/) where helpful.
     Each subsection should follow: description → danger/difficulty level → counter-technique or note -->

### 2.1 Pretraining data and the prior

Training data is generated, not collected. The prior favors simple structural causal models (SCMs). The recipe samples a hypothesis, then a synthetic dataset from it, and trains with cross-entropy on held-out rows of those datasets (Hollmann et al., 2023). An SCM is a set of variables linked by a causal graph, each computed from its parents plus noise. This pretraining happens once; users never repeat it.

Models differ here. Mitra mixes SCM and tree-ensemble priors, TabDPT shows real-data pretraining can compete, and Real-TabPFN found that continued pretraining on real data improves TabPFNv2 (Qu et al., 2026). TabICLv2 adds its own data generator built for diversity (Qu et al., 2026).

### 2.2 Architecture and encoding

TabPFN v1 treats each row as one token. TabPFNv2 gives every cell its own representation and alternates attention across rows and across columns (Qu et al., 2026). That design scales poorly with table size (Loza et al., 2026). TabICL uses two stages: column-then-row attention builds fixed-size row embeddings, then ICL runs over those embeddings, giving O(n² + nm²) cost (Qu et al., 2026). TabICLv2 adds a scalable softmax so attention does not flatten on long contexts (Qu et al., 2026). TabPFN-3 adds a retrieval-style classification decoder, row chunking and multi-query attention to cut memory (Loza et al., 2026, describing Grinsztajn et al., 2026).

### 2.3 Inference

The labeled training set is passed as context and test rows are predicted without gradient updates (Loza et al., 2026). Cross-validation is still useful for evaluation, but there is no per-dataset training loop to run.

### 2.4 Output

The network outputs an approximation of the PPD, a distribution over the label rather than a bare point estimate (Hollmann et al., 2023). Classification yields class probabilities. Current models also handle regression (Qu et al., 2026). This page does not cover how regression outputs are represented.

![TFM.png](../assets/TFM.png)

---
## 3. Advantages & Limitations

| | Aspect | Commentary |
|--|--------|------------|
| 🟢 | **No per-dataset fit-and-tune loop** | The TabPFN-3 authors report that a single forward pass beats all other models on TabArena, including tuned and ensembled baselines (Grinsztajn et al., 2026). The TabICLv2 authors report that, untuned, it beats a tuned, ensembled, fine-tuned RealTabPFN-2.5 (Qu et al., 2026). Both are authors’ own results. |
| 🟢 | **Strongest where trees and tuning are noisiest (in small data)** | TabArena reports foundation models excelling on smaller datasets (Erickson et al., 2025). Caveat: TabArena shares authors with the TabPFN team and with TabICLv2, so it is not independent of either camp. Exact ranks and Elo scores are on the TabArena leaderboard and are not reproduced here. |
| 🟢 | **Predictions are distributions by construction** | A default GBDT returns a point estimate or an uncalibrated score unless you add calibration or quantile objectives. No independent calibration benchmark is cited here, so how well calibrated these distributions are remains unverified. |
| 🟢 | **Fast end to end on small and mid-sized data** | On small data, the original TabPFN reports up to 230x speedup on CPU and 5,700x on GPU against AutoML systems it matched in accuracy, on datasets with up to 1,000 training rows, 100 numerical features and 10 classes (Hollmann et al., 2023). For mid-sized data, the TabICLv2 README states that 50,000 rows by 100 features are fit and predicted in under 10 seconds on an H100, about 10x faster than TabPFN-2.5 (Qu et al., 2026). That second comparison is against another TFM, not against AutoML. Both are the developers’ own measurements, and the 5,700x figure and the TabICLv2 timing assume a GPU. |
| 🟢 | **The large-data gap is closing** | TabPFN-3 reports beating default and 8-hour-tuned GBDTs on 13 large datasets, up to 1M training rows and 200 features (Grinsztajn et al., 2026). Authors’ benchmark only, and only 13 datasets. |
| 🔴 | **Size caps that trees do not have** | TabPFN v2 supports up to 10,000 samples (Loza et al., 2026). TabPFN-3 is benchmarked at up to 1M rows at 200 features, 100k rows at 2,000 features, or 1k rows at 20,000 features (Grinsztajn et al., 2026). TabICLv2 sources disagree on its pretraining range: the docs say 300 to 60K samples, while the README says 300 to 48K (Qu et al., 2026). TabArena leaves out very large data, with 250,000 training samples as its example (Erickson et al., 2025), so large-data claims rest mostly on authors’ benchmarks. |
| 🔴 | **Heavier inference** | An independent study found the highest-performing TFMs are often the least scalable, with some needing chunking, subsampling or hard row limits even on a 47GB GPU (Loza et al., 2026). The model re-reads your training set as context at prediction time. TabPFN-3 reports a 7 GiB KV cache per estimator at 1M rows (Grinsztajn et al., 2026). A fitted GBDT predicts cheaply on CPU. |
| 🔴 | **Feature types and widths** | The original TabPFN handled at most 1,000 training rows and 100 numerical features and was weak on categorical data (Loza et al., 2026, citing other work). TabICLv2 was pretrained on 2 to 100 columns (Qu et al., 2026). Text columns need specialized variants such as TabPFN-3-Plus (Grinsztajn et al., 2026). No source cited here tests high-cardinality categoricals on current models. |
| 🔴 | **Distribution shift** | Across nine TFMs and three TableShift datasets, every model degraded under shift, with gaps from 0.003 to 0.060 ROC-AUC and no robustness gain from any pretraining style (Loza et al., 2026). That study is small and does not compare against GBDTs, so it cannot say whether TFMs are worse than trees here. |


---
## 4. Key Takeaways
<div style="font-size: 16px; line-height: 1.6;">

- **A TFM replaces per-dataset training and tuning with a single forward pass over your labeled rows.**
- **It is pretrained on synthetic tables drawn from a prior, mostly structural causal models, though some variants add real data.**
- **Its output approximates a posterior predictive distribution, so uncertainty comes built in.**
- **Benchmarks favor TFMs on small and medium data, but most numbers come from the developers or closely related authors.**
- **Size limits, GPU inference cost, interpretability and distribution shift remain the main reasons to keep GBDTs in your toolbox.**
</div>

---
## 5. Sources & Verification Notes

**Papers**

Hollmann, N., Müller, S., Eggensperger, K. & Hutter, F. (2023). *TabPFN: A Transformer That Solves Small Tabular Classification Problems in a Second.* ICLR 2023. arXiv:2207.01848. https://arxiv.org/abs/2207.01848

Erickson, N. et al. (2025). *TabArena: A Living Benchmark for Machine Learning on Tabular Data.* NeurIPS 2025. arXiv:2506.16791. https://arxiv.org/abs/2506.16791

Qu, J., Holzmüller, D., Varoquaux, G. & Le Morvan, M. (2026). *TabICLv2: A better, faster, scalable, and open tabular foundation model.* arXiv:2602.11139. https://arxiv.org/abs/2602.11139

Grinsztajn, L. et al. (2026). *TabPFN-3: Technical Report.* Prior Labs. arXiv:2605.13986. https://arxiv.org/abs/2605.13986

Loza, M., Chushig-Muzo, D., Milara, E., Bote-Curiel, L., Estrada-Petrocelli, L. & Grijalva, F. (2026). *Empirical Evaluation of Out-Of-Distribution Performance of Tabular Foundation Models.* arXiv:2607.26000. https://arxiv.org/abs/2607.26000

**Official documentation**

TabICL project. *README and documentation* (speed and pretraining-range statements). https://github.com/soda-inria/tabicl and https://tabicl.readthedocs.io
